In [7]:
import json
import numpy as np
import plotly.graph_objects as go

# ==================================================
# Load data
# ==================================================
with open("triangulated-land.json", "r") as f:
    data = json.load(f)

pts_all = data["pts"]
triangles_all = data["triangles"]

# ==================================================
# Parameters
# ==================================================
R = 1.0
min_lat_deg = -60
min_lat = np.deg2rad(min_lat_deg)

sphere_radius = R
grat_radius   = 1.006 * R   # above sphere, below land
land_radius   = 1.008 * R   # above graticule

# ==================================================
# Helpers
# ==================================================
def sph_to_cart(lon, lat, r=1.0):
    lon = np.asarray(lon)
    lat = np.asarray(lat)
    x = r * np.cos(lat) * np.cos(lon)
    y = r * np.cos(lat) * np.sin(lon)
    z = r * np.sin(lat)
    return x, y, z

# ==================================================
# Build land mesh from indexed triangles
# ==================================================
X_land, Y_land, Z_land = [], [], []
I_land, J_land, K_land = [], [], []

vertex_offset = 0

for pts_k, tris_k in zip(pts_all, triangles_all):
    xyz_k = [sph_to_cart(lon, lat, land_radius) for lon, lat, elev in pts_k]
    lats_k = [lat for lon, lat, elev in pts_k]

    for x, y, z in xyz_k:
        X_land.append(float(x))
        Y_land.append(float(y))
        Z_land.append(float(z))

    for a, b, c in tris_k:
        # Drop triangles touching Antarctica / below cutoff
        if lats_k[a] < min_lat or lats_k[b] < min_lat or lats_k[c] < min_lat:
            continue

        I_land.append(vertex_offset + a)
        J_land.append(vertex_offset + b)
        K_land.append(vertex_offset + c)

    vertex_offset += len(pts_k)

# ==================================================
# Sphere surface
# ==================================================
u = np.linspace(-np.pi, np.pi, 240)
v = np.linspace(-np.pi/2, np.pi/2, 120)
uu, vv = np.meshgrid(u, v)

xs = sphere_radius * np.cos(vv) * np.cos(uu)
ys = sphere_radius * np.cos(vv) * np.sin(uu)
zs = sphere_radius * np.sin(vv)

# ==================================================
# Create figure
# ==================================================
fig = go.Figure()

# --------------------------------------------------
# Solid sphere
# --------------------------------------------------
fig.add_trace(go.Surface(
    x=xs,
    y=ys,
    z=zs,
    showscale=False,
    colorscale=[[0.0, "rgb(205,220,235)"], [1.0, "rgb(205,220,235)"]],
    opacity=1.0,
    hoverinfo="skip",
    lighting=dict(
        ambient=0.8,
        diffuse=0.5,
        specular=0.05,
        roughness=0.9,
        fresnel=0.05
    )
))

# --------------------------------------------------
# 10-degree graticule
# --------------------------------------------------
lat_vals = np.linspace(-np.pi/2, np.pi/2, 1500)
lon_vals = np.linspace(-np.pi, np.pi, 1700)

# Meridians every 10 degrees
for lon_deg in range(-180, 181, 10):
    lon = np.deg2rad(lon_deg)
    x, y, z = sph_to_cart(np.full_like(lat_vals, lon), lat_vals, grat_radius)

    width = 3 if lon_deg % 30 == 0 else 1
    color = "rgba(70,70,70,0.75)" if lon_deg % 30 == 0 else "rgba(90,90,90,0.45)"

    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode="lines",
        line=dict(color=color, width=width),
        hoverinfo="skip",
        showlegend=False
    ))

# Parallels every 10 degrees
for lat_deg in range(-80, 81, 10):
    lat = np.deg2rad(lat_deg)
    x, y, z = sph_to_cart(lon_vals, np.full_like(lon_vals, lat), grat_radius)

    width = 3 if lat_deg % 30 == 0 else 1
    color = "rgba(70,70,70,0.75)" if lat_deg % 30 == 0 else "rgba(90,90,90,0.45)"

    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode="lines",
        line=dict(color=color, width=width),
        hoverinfo="skip",
        showlegend=False
    ))

# --------------------------------------------------
# Land mesh
# --------------------------------------------------
fig.add_trace(go.Mesh3d(
    x=X_land,
    y=Y_land,
    z=Z_land,
    i=I_land,
    j=J_land,
    k=K_land,
    color="tan",
    flatshading=True,
    opacity=1.0,
    hoverinfo="skip",
    lighting=dict(
        ambient=0.9,
        diffuse=0.6,
        specular=0.05,
        roughness=1.0,
        fresnel=0.02
    )
))

# ==================================================
# Layout
# ==================================================
fig.update_layout(
    scene=dict(
        aspectmode="data",
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        camera=dict(
            eye=dict(x=1.6, y=1.6, z=1.0)
        ),
        bgcolor="white"
    ),
    margin=dict(l=0, r=0, t=0, b=0),
    showlegend=False
)

fig.show()